# 01-01_ログ一括チェック
## 目的
- ルータの状態確認コマンド実行し、確認観点に従ってチェックするためのノートブック

## 改修予定
- HTML出力部分を関数化する
- device_typeを使えるようにしたい -> Ciscoならそのまま自動的にenableまで実行

# モジュールのインポート

In [ ]:
# ==========================================
# 1. ライブラリのインポート
# ==========================================
# 標準ライブラリ・外部ライブラリ
import sys
import os
import json
from datetime import datetime
from netmiko import ConnectHandler
from getpass import getpass

# 自作モジュール（ネットワーク接続・実行管理用）
from module.login_utils import login_telnet, login_jump_ssh, enable_mode
from module.exe_utils import load_commands, execute_commands, prepare_and_map_results

print("="*50)
print(f"[{datetime.now().strftime('%H:%M:%S')}] [INFO]: 必要なモジュールの読み込みが完了しました。")
print("="*50)

# ユーザー事前定義

In [ ]:
# ==========================================
# 2. 実行対象と設定ファイルの定義
# ==========================================

# 接続対象の指定
jump_server_key = "clab-shuttle01" # 踏み台サーバの識別キー
target_key = "clab-dev-vmx01"      # ターゲットルータの識別キー

# 制御用設定
no_page = "set cli screen-length 0" # ページング無効化コマンド（IOS-XEの場合、terminal length 0）

# 入力ファイルパスの指定
device_file = "data/device_info.json" # デバイス接続情報
cmd_file = "input/commands.txt"       # 実行コマンドリスト

print("="*50)
print(f"[{datetime.now().strftime('%H:%M:%S')}] [INFO]: パラメータ定義をロードしました。")
print(f"  - 踏み台サーバ　　: {jump_server_key}")
print(f"  - ターゲット機器　: {target_key}")
print(f"  - 接続情報　　　　: {device_file}")
print(f"  - コマンド群　　　: {cmd_file}")
print("="*50)

# デバイス情報

In [ ]:
# ==========================================
# 3. 接続情報の展開と実行準備
# ==========================================

# --- 構成情報の読み込み ---
# JSONから指定したホストの接続パラメータ（IP、ポート、OS種別等）を抽出
with open(device_file, 'r') as f:
    device_info = json.load(f)

device_jump = device_info[jump_server_key]    # 踏み台サーバのプロファイル
device_target = device_info[target_key]      # ターゲットルータのプロファイル

# --- ログ出力設定 ---
# 作業エビデンスを保存するディレクトリとファイル名の生成
ticket_no = input("Enter [作業チケット番号] (例: TKT12345): ")
os.makedirs("output", exist_ok=True)
log_name = f"output/{ticket_no}_{target_key}_{datetime.now().strftime('%Y%m%d')}.log"

# --- 認証情報の補完 ---
# JSONにパスワードが未記載の場合、インタラクティブに入力を促す
if not device_jump.get('password'):
    device_jump['password'] = getpass(f"Enter 踏み台({jump_server_key}) のパスワード: ")

if not device_target.get('password'):
    device_target['password'] = getpass(f"Enter ターゲット({target_key}) のパスワード: ")

# 特権モード用のパスワード設定（必要な場合）
if not device_target.get('secret'):
    device_target['secret'] = getpass(f"Enter ターゲット({target_key}) の enableパスワード: ")

print("="*50)
print(f"[{datetime.now().strftime('%H:%M:%S')}] [INFO]: 準備が整いました。")
print(f"  - 接続ルート　: {jump_server_key}({device_jump['host']}) ->  {target_key}({device_target['host']})")
print(f"  - 保存先ログ　: {log_name}")
print("="*50)
print("⚠️  [CHECK]: 接続ルートが意図した踏み台サーバ・機器か、上記を確認してください。")

# ログイン

In [ ]:
# ==========================================
# 4. ネットワーク機器への接続実行
# ==========================================

# --- 踏み台サーバへのログイン ---
# Telnetを使用して中継地点(Jump Server)に接続
print(f"[{datetime.now().strftime('%H:%M:%S')}] [RUN]: 踏み台サーバ ({jump_server_key}) へTelnet接続中...")
conn = login_telnet(device_jump, jump_server_key, log_name)

# --- ターゲットデバイスへのログイン ---
# 踏み台のセッションを維持したまま、SSHで最終目的地へ接続
print(f"[{datetime.now().strftime('%H:%M:%S')}] [RUN]: ターゲット ({target_key}) へSSH接続を試行します...")
conn = login_jump_ssh(
    conn, 
    device_target["host"], 
    device_target["username"], 
    device_target["password"], 
    label=target_key
)

# --- 特権モードへの移行 (必要な場合) ---
# if device_target.get("secret"):
#     print(f"[RUN]: 特権モード(enable)に移行します。")
#     enable_mode(conn, device_target["secret"])

# --- 接続完了確認 ---
print("="*50)
print(f"[{datetime.now().strftime('%H:%M:%S')}] [SUCCESS]: ログイン処理が完了しました。")
print(f"現在のプロンプト: {conn.find_prompt()}")
print("="*50)
print("⚠️  [CHECK]: プロンプトが意図した機器のものか、上記を確認してください。")

# コマンド実行

In [ ]:
# ==========================================
# 5. コマンド実行と実行結果の整形
# ==========================================

# --- 制御設定の投入 ---
# ページング（--More--）を無効化し、出力を一括取得できるように設定
print(f"[{datetime.now().strftime('%H:%M:%S')}] [RUN]: ページング無効コマンドを実行中... ({no_page})")
conn.send_command(no_page)

# --- コマンドの読み込み ---
# ファイルから実行対象のコマンドリストをロード
raw_lines, cmd_list_for_device = load_commands(cmd_file)
print(f"[{datetime.now().strftime('%H:%M:%S')}] [INFO]: コマンドファイルを読み込みました。 (実行件数: {len(cmd_list_for_device)}件)")

# --- コマンドの逐次実行 ---
# 実機に対してコマンドを投入し、生の応答を取得
print(f"[{datetime.now().strftime('%H:%M:%S')}] [RUN]: コマンドを実行しています。しばらくお待ちください...")
results_map = execute_commands(conn, cmd_list_for_device)

# --- 結果のパースとマッピング ---
# 実行したコマンドと、その応答結果を対応表(dict)に整理
outputs = prepare_and_map_results(raw_lines, results_map)

print("=" * 50)
print(f"[{datetime.now().strftime('%H:%M:%S')}] [SUCCESS]: 全コマンドの実行と結果の紐付けが完了しました。")
print(f"  - 処理済みコマンド数　: {len(outputs)} 件")
print("=" * 50)

# 実行結果の確認

In [ ]:
from IPython.display import display, HTML
import html

# outputs が存在することを確認
if 'outputs' in locals() and outputs:
    css_and_js = """
    <style>
        .result-container { margin-bottom: 30px; border: 2px solid #4A90E2; border-radius: 8px; overflow: hidden; font-family: sans-serif; box-shadow: 0 2px 5px rgba(0,0,0,0.1); transition: all 0.3s; }
        .result-header { background-color: #4A90E2; color: white; padding: 12px; font-weight: bold; font-size: 15px; }
        .info-area { background-color: #f0f7ff; padding: 15px; border-bottom: 1px solid #ddd; }
        .viewpoint-box { margin-bottom: 8px; font-size: 14px; color: #333; }
        .keyword-badge { font-size: 13px; color: #d9534f; background: #fff; padding: 2px 8px; border: 1px solid #d9534f; border-radius: 12px; display: inline-block; }
        
        .log-frame { 
            background-color: #1e1e1e; color: #d4d4d4; padding: 10px 0; 
            max-height: 400px; overflow-y: auto; 
            font-family: 'Consolas', 'Monaco', monospace; font-size: 13px; line-height: 1.6;
        }
        /* 行全体のスタイル */
        .log-line { padding: 0 15px; white-space: pre; }
        /* ハイライト行のスタイル */
        .highlight-line { background-color: #555500; color: #ffffff; font-weight: bold; display: block; width: 100%; }
        .highlight-line mark { background: transparent; color: #ffeb3b; text-decoration: underline; }

        /* チェック時のグレーアウト */
        .checked-item { opacity: 0.3; filter: grayscale(1); }

        .check-area { padding: 15px; background: #f9f9f9; border-top: 1px solid #eee; }
        .check-label { cursor: pointer; display: flex; align-items: center; gap: 10px; font-weight: bold; font-size: 14px; color: #444; }
        input[type="checkbox"] { width: 18px; height: 18px; }

        #bottom-status-area {
            background-color: #f8f9fa; border: 2px solid #ddd; padding: 20px;
            margin-top: 40px; margin-bottom: 60px; border-radius: 12px; text-align: center;
        }
        .all-done { border-color: #28a745 !important; background-color: #e9f7ef !important; }
        .completion-msg { display: none; color: #28a745; font-size: 20px; font-weight: bold; margin-top: 10px; }
    </style>

    <script>
    function updateProgress(idx) {
        const container = document.getElementById('box-' + idx);
        const isChecked = document.getElementById('check-' + idx).checked;
        
        if (isChecked) container.classList.add('checked-item');
        else container.classList.remove('checked-item');

        const boxes = document.querySelectorAll('.check-input');
        const checkedCount = Array.from(boxes).filter(cb => cb.checked).length;
        document.getElementById('current-count').innerText = checkedCount;
        
        const statusArea = document.getElementById('bottom-status-area');
        if (checkedCount === boxes.length && boxes.length > 0) {
            statusArea.classList.add('all-done');
            document.getElementById('complete-msg-text').style.display = 'block';
        } else {
            statusArea.classList.remove('all-done');
            document.getElementById('complete-msg-text').style.display = 'none';
        }
    }
    </script>
    """
    display(HTML(css_and_js))

    for i, (full_line, out) in enumerate(outputs.items()):
        parts = full_line.split(';;', 2)
        cmd = parts[0].strip()
        viewpoint = parts[1].strip() if len(parts) > 1 else "確認事項なし"
        keyword = parts[2].strip() if len(parts) > 2 else ""

        # 行ごとに処理
        lines = out.splitlines()
        processed_lines = []
        
        for line in lines:
            safe_line = html.escape(line)
            # 改善点: キーワードが含まれる場合、行全体にクラスを付与
            if keyword and keyword.lower() in line.lower():
                # キーワード部分だけさらにmarkタグで強調（任意）
                highlighted = safe_line.replace(html.escape(keyword), f"<mark>{html.escape(keyword)}</mark>")
                processed_lines.append(f'<div class="log-line highlight-line">{highlighted}</div>')
            else:
                processed_lines.append(f'<div class="log-line">{safe_line}</div>')
        
        display_out = "".join(processed_lines)

        html_content = f"""
        <div class="result-container" id="box-{i}">
            <div class="result-header">No.{i+1} : {cmd}</div>
            <div class="info-area">
                <div class="viewpoint-box"><b>📌 確認観点:</b> {viewpoint}</div>
                {f"<div class='keyword-badge'>🔍 強調ワード: {html.escape(keyword)}</div>" if keyword else ""}
            </div>
            <div class="log-frame">{display_out}</div>
            <div class="check-area">
                <label class="check-label">
                    <input type="checkbox" id="check-{i}" class="check-input" onchange="updateProgress({i})">
                    この項目の目視確認を完了しました
                </label>
            </div>
        </div>
        """
        display(HTML(html_content))

    display(HTML(f"""
        <div id="bottom-status-area">
            <div style="font-size: 18px; font-weight: bold; color: #555;">
                📊 確認状況: <span id="current-count">0</span> / {len(outputs)} 完了
            </div>
            <div id="complete-msg-text" class="completion-msg">
                ✅ すべての項目の目視確認が完了しました！
            </div>
        </div>
    """))
else:
    print("=" * 50)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] [ERROR]: 表示可能な実行結果(outputs)が見つかりません。")
    print("=" * 50)

# ログアウト

In [ ]:
# --- 接続終了 ---
conn.disconnect()

print("="*50)
print(f"[{datetime.now().strftime('%H:%M:%S')}] [SUCCESS]: ログアウト処理が完了しました。")
print("="*50)